In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [3]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Record count: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

Record count: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+-------+-----------+-------+-------------------+-------+-------+
|amount |category   |store  |timestamp          |tx_id  |user_id|
+-------+-----------+-------+-------------------+-------+-------+
|129.93 |clothing   |Gdańsk |2026-01-15 10:54:36|TX00001|u029   |
|3683.67|books      |Warsaw |2026-01-15 09:00:57|TX00002|u036   |
|473.01 |electronics|Kraków |2026-01-15 08:08:40|TX00003|u008   |
|3581.52|clothing   |Wrocław|2026-01-15 10:33:15|TX00004|u051   |
|1394.56|clothing   |Warsaw |2026-01-15 10:02:39|TX00005|u151   |
|1392.97|food       |Kraków |2026-01-15 09:55:24|TX00006|u088   |
|1902.74|food       |Gdańsk |2026-01-15 08:27:54|TX00007|u024   |
|4036.61|electronics|Wrocław|2026-01-15 10:44:51|TX00008|u068   |
|2762.44|clothin

In [4]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(avg("amount"), 2).alias("avg_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+-------+--------+----------+-------+
|  store|tx_count| total_PLN|avg_PLN|
+-------+--------+----------+-------+
| Gdańsk|    2516| 6334363.9|2517.63|
| Kraków|    2477|6145724.95|2481.12|
| Warsaw|    2497|6205548.34| 2485.2|
|Wrocław|    2510|6249236.46|2489.74|
+-------+--------+----------+-------+



In [6]:
from pyspark.sql.functions import min as _min, max as _max

category_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)
category_summary.show()

+-----------+--------+----------+-------+-------+
|   category|tx_count| total_PLN|min_PLN|max_PLN|
+-----------+--------+----------+-------+-------+
|      books|    2584|6452522.35|   7.31|4994.19|
|   clothing|    2483|6153393.79|    9.8|4995.97|
|electronics|    2499|6230167.13|   5.35|4997.71|
|       food|    2434|6098790.38|   7.71|4999.94|
+-----------+--------+----------+-------+-------+



In [7]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

# Output đẹp hơn — tách window.start và window.end
(
    hourly
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .show(truncate=False)
)

+------------------------------------------+--------+----------+
|window                                    |tx_count|total_PLN |
+------------------------------------------+--------+----------+
|{2026-01-15 08:00:00, 2026-01-15 09:00:00}|3297    |8223061.29|
|{2026-01-15 09:00:00, 2026-01-15 10:00:00}|3311    |8222044.08|
|{2026-01-15 10:00:00, 2026-01-15 11:00:00}|3392    |8489768.28|
+------------------------------------------+--------+----------+

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-01-15 08:00:00|2026-01-15 09:00:00|3297    |8223061.29|
|2026-01-15 09:00:00|2026-01-15 10:00:00|3311    |8222044.08|
|2026-01-15 10:00:00|2026-01-15 11:00:00|3392    |8489768.28|
+-------------------+-------------------+--------+----------+



In [8]:
store_30min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "store",
        "tx_count",
        "total_PLN",
    )
    .orderBy("from", "store")
)
store_30min.show(truncate=False)

+-------------------+-------------------+-------+--------+----------+
|from               |to                 |store  |tx_count|total_PLN |
+-------------------+-------------------+-------+--------+----------+
|2026-01-15 08:00:00|2026-01-15 08:30:00|Gdańsk |405     |1028504.05|
|2026-01-15 08:00:00|2026-01-15 08:30:00|Kraków |409     |1017576.75|
|2026-01-15 08:00:00|2026-01-15 08:30:00|Warsaw |440     |1101535.19|
|2026-01-15 08:00:00|2026-01-15 08:30:00|Wrocław|397     |1019272.87|
|2026-01-15 08:30:00|2026-01-15 09:00:00|Gdańsk |432     |1068555.11|
|2026-01-15 08:30:00|2026-01-15 09:00:00|Kraków |411     |964363.97 |
|2026-01-15 08:30:00|2026-01-15 09:00:00|Warsaw |379     |910122.22 |
|2026-01-15 08:30:00|2026-01-15 09:00:00|Wrocław|424     |1113131.13|
|2026-01-15 09:00:00|2026-01-15 09:30:00|Gdańsk |433     |1081695.24|
|2026-01-15 09:00:00|2026-01-15 09:30:00|Kraków |397     |961613.06 |
|2026-01-15 09:00:00|2026-01-15 09:30:00|Warsaw |387     |975593.4  |
|2026-01-15 09:00:00

In [9]:
from pyspark.sql.functions import desc

krakow_hourly = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("total_PLN"))
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "total_PLN",
    )
    .orderBy(desc("total_PLN"))
)
krakow_hourly.show(truncate=False)

+-------------------+-------------------+----------+
|from               |to                 |total_PLN |
+-------------------+-------------------+----------+
|2026-01-15 10:00:00|2026-01-15 11:00:00|2105757.4 |
|2026-01-15 09:00:00|2026-01-15 10:00:00|2058026.83|
|2026-01-15 08:00:00|2026-01-15 09:00:00|1981940.72|
+-------------------+-------------------+----------+



In [10]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .orderBy("from")
)
sliding.show(truncate=False)

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-01-15 07:30:00|2026-01-15 08:30:00|1651    |4166888.86|
|2026-01-15 08:00:00|2026-01-15 09:00:00|3297    |8223061.29|
|2026-01-15 08:30:00|2026-01-15 09:30:00|3282    |8120222.29|
|2026-01-15 09:00:00|2026-01-15 10:00:00|3311    |8222044.08|
|2026-01-15 09:30:00|2026-01-15 10:30:00|3360    |8382383.45|
|2026-01-15 10:00:00|2026-01-15 11:00:00|3392    |8489768.28|
|2026-01-15 10:30:00|2026-01-15 11:30:00|1707    |4265379.05|
+-------------------+-------------------+--------+----------+



In [11]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):         {tumbling_rows} windows")
print(f"Sliding  (1h / 30min): {sliding_rows} windows")

# YOUR ANSWER:
# Sliding tạo nhiều rows hơn vì các window CHỒNG LẤN nhau.
# Tumbling: mỗi tin nhắn thuộc đúng 1 window (08:00-09:00, 09:00-10:00, 10:00-11:00) → 3 windows.
# Sliding với bước 30 phút: cứ mỗi 30 phút lại bắt đầu 1 window mới (08:00-09:00, 08:30-09:30,
# 09:00-10:00, 09:30-10:30, 10:00-11:00) → 5 windows. Mỗi giao dịch xuất hiện trong 2 windows
# (trừ các giao dịch ở rìa).

Tumbling (1h):         3 windows
Sliding  (1h / 30min): 7 windows


In [12]:
# Homework 1: Find the hour in which store "Gdańsk" had the lowest average transaction amount.

gdansk_avg_hourly = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("avg_PLN"))
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "avg_PLN",
    )
    .orderBy("avg_PLN")     # ascending — lowest first
)
gdansk_avg_hourly.show(truncate=False)

print("→ The hour with the LOWEST average for Gdańsk is the FIRST row above.")

+-------------------+-------------------+-------+
|from               |to                 |avg_PLN|
+-------------------+-------------------+-------+
|2026-01-15 08:00:00|2026-01-15 09:00:00|2505.45|
|2026-01-15 10:00:00|2026-01-15 11:00:00|2519.87|
|2026-01-15 09:00:00|2026-01-15 10:00:00|2527.57|
+-------------------+-------------------+-------+

→ The hour with the LOWEST average for Gdańsk is the FIRST row above.


In [14]:
# Homework 2: Count how many transactions per category occurred in the 09:00–09:30 window.

from pyspark.sql.functions import hour, minute

hw2_safe = (
    df.filter((hour("timestamp") == 9) & (minute("timestamp") < 30))
    .groupBy("category")
    .agg(count("tx_id").alias("tx_count"))
    .orderBy(desc("tx_count"))
)
hw2_safe.show()

+-----------+--------+
|   category|tx_count|
+-----------+--------+
|   clothing|     420|
|      books|     410|
|electronics|     410|
|       food|     396|
+-----------+--------+



In [15]:
# Homework 3: Use a 15-minute window and find which quarter-hour had the peak transaction volume.

quarter_hourly = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("tx_count"))
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
    )
    .orderBy(desc("tx_count"))
)
quarter_hourly.show(truncate=False)

print("→ The PEAK quarter-hour is the FIRST row above.")

+-------------------+-------------------+--------+
|from               |to                 |tx_count|
+-------------------+-------------------+--------+
|2026-01-15 10:30:00|2026-01-15 10:45:00|875     |
|2026-01-15 10:00:00|2026-01-15 10:15:00|858     |
|2026-01-15 09:45:00|2026-01-15 10:00:00|854     |
|2026-01-15 08:00:00|2026-01-15 08:15:00|844     |
|2026-01-15 10:45:00|2026-01-15 11:00:00|832     |
|2026-01-15 08:30:00|2026-01-15 08:45:00|829     |
|2026-01-15 10:15:00|2026-01-15 10:30:00|827     |
|2026-01-15 09:00:00|2026-01-15 09:15:00|823     |
|2026-01-15 09:30:00|2026-01-15 09:45:00|821     |
|2026-01-15 08:45:00|2026-01-15 09:00:00|817     |
|2026-01-15 09:15:00|2026-01-15 09:30:00|813     |
|2026-01-15 08:15:00|2026-01-15 08:30:00|807     |
+-------------------+-------------------+--------+

→ The PEAK quarter-hour is the FIRST row above.


In [16]:
spark.stop()
print("Spark stopped.")

Spark stopped.
